In [1]:
import datetime
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import json

from sklearn import datasets, ensemble, model_selection
from scipy.stats import anderson_ksamp

In [2]:
content = requests.get("https://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip").content
with zipfile.ZipFile(io.BytesIO(content)) as arc:
    raw_data = pd.read_csv(arc.open("hour.csv"), header=0, sep=',', parse_dates=['dteday'])

In [3]:
raw_data.index = raw_data.apply(lambda row: datetime.datetime.combine(row.dteday.date(), datetime.time(row.hr)),
                                axis=1)

In [4]:
raw_data.head()


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
2011-01-01 00:00:00,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
2011-01-01 01:00:00,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2011-01-01 02:00:00,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
2011-01-01 03:00:00,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
2011-01-01 04:00:00,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [5]:
raw_data.info()


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 17379 entries, 2011-01-01 00:00:00 to 2012-12-31 23:00:00
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   instant     17379 non-null  int64         
 1   dteday      17379 non-null  datetime64[ns]
 2   season      17379 non-null  int64         
 3   yr          17379 non-null  int64         
 4   mnth        17379 non-null  int64         
 5   hr          17379 non-null  int64         
 6   holiday     17379 non-null  int64         
 7   weekday     17379 non-null  int64         
 8   workingday  17379 non-null  int64         
 9   weathersit  17379 non-null  int64         
 10  temp        17379 non-null  float64       
 11  atemp       17379 non-null  float64       
 12  hum         17379 non-null  float64       
 13  windspeed   17379 non-null  float64       
 14  casual      17379 non-null  int64         
 15  registered  17379 non-null  int64  

In [6]:
from scipy import stats
import random

#Significance level

p_value = 0.05
rejected = 0

In [7]:
numerical_features = ['temp', 'atemp', 'hum', 'windspeed', 'mnth', 'hr', 'weekday']
categorical_features = ['season', 'holiday', 'workingday']

In [8]:
reference = raw_data.loc['2011-01-01 00:00:00':'2011-01-28 23:00:00']
current = raw_data.loc['2011-01-29 00:00:00':'2011-02-28 23:00:00']

In [9]:
col = 'temp'
stats.ks_2samp(reference[col], current[col])

KstestResult(statistic=0.3268630919426928, pvalue=6.026276035162504e-32, statistic_location=0.22, statistic_sign=1)

In [10]:
for col in numerical_features:
    test = stats.ks_2samp(reference[col], current[col])

    print(col, test)
    if test[1] < p_value:
        rejected += 1
        print("Column rejected", col)

print("We rejected ",rejected," columns in total out of {} columns".format(len(numerical_features)))

temp KstestResult(statistic=0.3268630919426928, pvalue=6.026276035162504e-32, statistic_location=0.22, statistic_sign=1)
Column rejected temp
atemp KstestResult(statistic=0.3233995435947986, pvalue=2.855460838157155e-31, statistic_location=0.2273, statistic_sign=1)
Column rejected atemp
hum KstestResult(statistic=0.10336407541938417, pvalue=0.0014918623865842317, statistic_location=0.35, statistic_sign=-1)
Column rejected hum
windspeed KstestResult(statistic=0.07590999725436713, pvalue=0.04055120974818981, statistic_location=0.3284, statistic_sign=1)
Column rejected windspeed
mnth KstestResult(statistic=0.9026425591098748, pvalue=1.472945287377825e-288, statistic_location=1, statistic_sign=1)
Column rejected mnth
hr KstestResult(statistic=0.011077053260776609, pvalue=0.999999999980103, statistic_location=3, statistic_sign=-1)
weekday KstestResult(statistic=0.022863019926092965, pvalue=0.9929320453958753, statistic_location=1, statistic_sign=-1)
We rejected  5  columns in total out of 7

In [11]:
from scipy.stats import chi2_contingency

def drift_chisquare(sample1, sample2):
    return chi2_contingency([sample1, sample2])[1]

for col in categorical_features:
    print(col, set(reference[col].values))

season {1}
holiday {0, 1}
workingday {0, 1}


In [12]:
for col in categorical_features:
    val = drift_chisquare(reference[col].value_counts(),current[col].value_counts() )

    print(col,val)
    rejected = 0
    if val < p_value:
        rejected += 1
        print("Column rejected", col)

print("We rejected ",rejected," columns in total out of {} columns".format(len(categorical_features)))

season 1.0
holiday 0.6986573626612528
workingday 0.5917879941201512
We rejected  0  columns in total out of 3 columns


In [13]:
target = 'cnt'
prediction = 'prediction'
numerical_features = ['temp', 'atemp', 'hum', 'windspeed', 'mnth', 'hr', 'weekday']
categorical_features = ['season', 'holiday', 'workingday', ]#'weathersit']

In [14]:
reference = raw_data.loc['2011-01-01 00:00:00':'2011-01-28 23:00:00']
current = raw_data.loc['2011-01-29 00:00:00':'2011-02-28 23:00:00']

In [15]:
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    reference[numerical_features + categorical_features],
    reference[target],
    test_size=0.3
)

In [16]:
regressor = ensemble.RandomForestRegressor(random_state = 0)

regressor.fit(X_train, y_train)

preds_test = regressor.predict(X_test)

In [17]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

ref_mae=mean_absolute_error(y_test,preds_test)
ref_mse=mean_squared_error(y_test,preds_test)
ref_r2 = r2_score(y_test,preds_test)


print("MAE",ref_mae)
print("MSE",ref_mse)
print("R2",ref_r2)

MAE 10.446021505376343
MSE 263.8750129032258
R2 0.8810282991810754


In [18]:
current_x=current[numerical_features + categorical_features]
current_y=current[target]

current_pred = regressor.predict(current_x)

In [19]:
print("MAE",mean_absolute_error(current_y,current_pred))
print("MSE",mean_squared_error(current_y,current_pred))

r2 = r2_score(current_y,current_pred)
print(r2)

MAE 18.941863699582754
MSE 967.9723020862308
0.7496096141013073


In [20]:
import mlflow
#import mlflow.sklearn
from mlflow.tracking import MlflowClient
import os

In [21]:
mlflow.set_experiment("Bicycle–Sharing")


2024/07/12 22:39:17 INFO mlflow.tracking.fluent: Experiment with name 'Bicycle–Sharing' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///Users/scaler/Documents/MLopsJune/mlruns/153511767200807310', creation_time=1720804157316, experiment_id='153511767200807310', last_update_time=1720804157316, lifecycle_stage='active', name='Bicycle–Sharing', tags={}>

In [22]:
with mlflow.start_run():

    mlflow.set_tag('mlflow.runName','Refrence_run')
    mlflow.log_metric("MAE",ref_mae)
    mlflow.log_metric("MSE",ref_mse)
    mlflow.log_metric("R2",ref_r2)

    mlflow.sklearn.log_model(regressor, "model")

/opt/anaconda3/lib/python3.11/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


In [ ]:
!mlflow ui -p 5024

[2024-07-12 22:42:09 +0530] [50498] [INFO] Starting gunicorn 22.0.0
[2024-07-12 22:42:09 +0530] [50498] [INFO] Listening at: http://127.0.0.1:5024 (50498)
[2024-07-12 22:42:09 +0530] [50498] [INFO] Using worker: sync
[2024-07-12 22:42:09 +0530] [50499] [INFO] Booting worker with pid: 50499
[2024-07-12 22:42:09 +0530] [50500] [INFO] Booting worker with pid: 50500
[2024-07-12 22:42:09 +0530] [50501] [INFO] Booting worker with pid: 50501
[2024-07-12 22:42:09 +0530] [50502] [INFO] Booting worker with pid: 50502


In [25]:
experiment_batches = [
    ('2011-01-29 00:00:00','2011-02-07 23:00:00'),
    ('2011-02-07 00:00:00','2011-02-14 23:00:00'),
    ('2011-02-15 00:00:00','2011-02-21 23:00:00'),
]


In [26]:
#start new run
for date in experiment_batches:
    with mlflow.start_run() as run: #inside brackets run_name='test'

        mlflow.set_tag('mlflow.runName',"run_"+str(date[0])+" : "+str(date[1]))

        # Log parameters
        mlflow.log_param("begin", date[0])
        mlflow.log_param("end", date[1])

        # Get metrics
        current_data=current.loc[date[0]:date[1]]
        current_x=current_data[numerical_features + categorical_features]
        current_y=current_data[target]
        current_pred = regressor.predict(current_x)

        mae=mean_absolute_error(current_y,current_pred)
        mse=mean_squared_error(current_y,current_pred)
        r2 = r2_score(current_y,current_pred)

        # Log metrics
        mlflow.log_metric('MAE', round(mae, 3))
        mlflow.log_metric('MSE', round(mse, 3))
        mlflow.log_metric('R2', round(r2, 3))

        print(run.info)

<RunInfo: artifact_uri='file:///Users/scaler/Documents/MLopsJune/mlruns/153511767200807310/77443fd25bf8421a8b34f99d8dcaa30d/artifacts', end_time=None, experiment_id='153511767200807310', lifecycle_stage='active', run_id='77443fd25bf8421a8b34f99d8dcaa30d', run_name='ambitious-loon-930', run_uuid='77443fd25bf8421a8b34f99d8dcaa30d', start_time=1720804312196, status='RUNNING', user_id='scaler'>
<RunInfo: artifact_uri='file:///Users/scaler/Documents/MLopsJune/mlruns/153511767200807310/3bb55135cb7e4a338b9187ee272c5a5c/artifacts', end_time=None, experiment_id='153511767200807310', lifecycle_stage='active', run_id='3bb55135cb7e4a338b9187ee272c5a5c', run_name='serious-bee-916', run_uuid='3bb55135cb7e4a338b9187ee272c5a5c', start_time=1720804312378, status='RUNNING', user_id='scaler'>
<RunInfo: artifact_uri='file:///Users/scaler/Documents/MLopsJune/mlruns/153511767200807310/4e19637ecdbf4bf591b8b232db5c7d47/artifacts', end_time=None, experiment_id='153511767200807310', lifecycle_stage='active', ru